<a href="https://colab.research.google.com/github/Jirtus-sanasam/MLP-Diabetes/blob/main/Diabetes_stacking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install lightgbm optuna -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score,
                              classification_report, confusion_matrix, roc_curve)
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

import warnings
warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.2 MB/s eta 0:00:00


In [3]:
df = pd.read_csv('/content/diabetes_data2.csv')

cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols] = df[cols].replace(0, np.nan)

knn_imputer = KNNImputer(n_neighbors=5)
df[cols] = knn_imputer.fit_transform(df[cols])

def cap_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = df[col].clip(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
    return df

for col in df.columns[:-1]:
    df = cap_outliers(df, col)

In [4]:
df['Glucose_BMI']           = df['Glucose'] * df['BMI']
df['Glucose_Insulin']       = df['Glucose'] * df['Insulin']
df['BMI_Age']               = df['BMI'] * df['Age']
df['Insulin_SkinThickness'] = df['Insulin'] * df['SkinThickness']
df['Glucose_Insulin_Ratio'] = df['Glucose'] / (df['Insulin'] + 1e-6)
df['BMI_Age_Ratio']         = df['BMI'] / (df['Age'] + 1e-6)
df['Glucose_sq']            = df['Glucose'] ** 2
df['BMI_sq']                = df['BMI'] ** 2
df['Age_sq']                = df['Age'] ** 2

df['Age_Group'] = pd.cut(df['Age'], bins=[0,30,45,60,100],
                          labels=['Young','Middle','Senior','Elderly'])
df = pd.get_dummies(df, columns=['Age_Group'], drop_first=True, dtype=int)

df['BMI_Category'] = pd.cut(df['BMI'], bins=[0,18.5,24.9,29.9,100],
                              labels=['Underweight','Normal','Overweight','Obese'])
df = pd.get_dummies(df, columns=['BMI_Category'], drop_first=True, dtype=int)

df['Glucose_Level'] = pd.cut(df['Glucose'], bins=[0,99,125,300],
                               labels=['Normal','Prediabetes','Diabetes'])
df = pd.get_dummies(df, columns=['Glucose_Level'], drop_first=True, dtype=int)

df['HOMA_IR']                      = (df['Glucose'] * df['Insulin']) / 405
df['High_Pregnancies']             = (df['Pregnancies'] > 3).astype(int)
df['Log_Insulin']                  = np.log1p(df['Insulin'])
df['Log_DiabetesPedigreeFunction'] = np.log1p(df['DiabetesPedigreeFunction'])
df['Risk_Score'] = (df['Glucose']/100) + (df['BMI']/25) + \
                   (df['Age']/50) + df['DiabetesPedigreeFunction']

for col in df.select_dtypes(include='bool').columns:
    df[col] = df[col].astype(int)

print("Dataset shape:", df.shape)

Dataset shape: (768, 31)


In [5]:
selected_features = ['Glucose', 'Glucose_BMI', 'Glucose_Insulin', 'BMI_Age',
                     'Insulin_SkinThickness', 'BMI_Age_Ratio', 'Glucose_sq',
                     'BMI_sq', 'HOMA_IR', 'Risk_Score']

X = df[selected_features]
Y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# SMOTE on training data
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

print("Before SMOTE:", dict(pd.Series(y_train).value_counts()))
print("After SMOTE :", dict(pd.Series(y_train_sm).value_counts()))

Before SMOTE: {0: np.int64(400), 1: np.int64(214)}
After SMOTE : {0: np.int64(400), 1: np.int64(400)}


In [6]:
# ── XGBoost ──────────────────────────────────────────
def build_xgb():
    return XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        eval_metric='logloss',
        use_label_encoder=False,
        random_state=42,
        n_jobs=-1
    )

# ── LightGBM ─────────────────────────────────────────
def build_lgbm():
    return LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

# ── Keras MLP ─────────────────────────────────────────
def build_mlp(input_dim):
    model = Sequential([
        Dense(128, activation='relu', input_shape=(input_dim,)),
        BatchNormalization(),
        Dropout(0.3),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [7]:
# ── XGBoost ──────────────────────────────────────────
def build_xgb():
    return XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        eval_metric='logloss',
        use_label_encoder=False,
        random_state=42,
        n_jobs=-1
    )

# ── LightGBM ─────────────────────────────────────────
def build_lgbm():
    return LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )

# ── Keras MLP ─────────────────────────────────────────
def build_mlp(input_dim):
    model = Sequential([
        Dense(128, activation='relu', input_shape=(input_dim,)),
        BatchNormalization(),
        Dropout(0.3),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [8]:
from sklearn.linear_model import LogisticRegression

def get_oof_predictions(X_data, y_data, n_splits=5):
    """
    Generate Out-Of-Fold predictions from all 3 base models.
    These become the input features for the meta-learner.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    oof_xgb  = np.zeros(len(y_data))
    oof_lgbm = np.zeros(len(y_data))
    oof_mlp  = np.zeros(len(y_data))

    X_arr = X_data
    Y_arr = y_data if isinstance(y_data, np.ndarray) else y_data.values

    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_arr, Y_arr)):
        print(f"\n--- Fold {fold+1}/{n_splits} ---")

        X_tr, X_val = X_arr[train_idx], X_arr[val_idx]
        y_tr, y_val = Y_arr[train_idx],  Y_arr[val_idx]

        # SMOTE inside each fold — no leakage
        smote = SMOTE(random_state=42)
        X_tr_sm, y_tr_sm = smote.fit_resample(X_tr, y_tr)

        # ── XGBoost ──
        xgb = build_xgb()
        xgb.fit(X_tr_sm, y_tr_sm, verbose=False)
        oof_xgb[val_idx] = xgb.predict_proba(X_val)[:, 1]

        # ── LightGBM ──
        lgbm = build_lgbm()
        lgbm.fit(X_tr_sm, y_tr_sm)
        oof_lgbm[val_idx] = lgbm.predict_proba(X_val)[:, 1]

        # ── MLP ──
        mlp = build_mlp(X_tr_sm.shape[1])
        early_stop = EarlyStopping(monitor='val_loss', patience=10,
                                   restore_best_weights=True)
        mlp.fit(X_tr_sm, y_tr_sm,
                epochs=100, batch_size=32,
                validation_data=(X_val, y_val),
                callbacks=[early_stop], verbose=0)
        oof_mlp[val_idx] = mlp.predict(X_val).flatten()

        # Fold metrics
        fold_auc = roc_auc_score(y_val,
                    (oof_xgb[val_idx] + oof_lgbm[val_idx] + oof_mlp[val_idx]) / 3)
        fold_metrics.append(fold_auc)
        print(f"  Avg Ensemble AUC (fold): {fold_auc:.4f}")

    print(f"\n✅ OOF Generation Complete")
    print(f"   Mean Fold AUC: {np.mean(fold_metrics):.4f} ± {np.std(fold_metrics):.4f}")

    return oof_xgb, oof_lgbm, oof_mlp


# Full scaled data for OOF
X_full_scaled = scaler.fit_transform(X)
Y_arr         = Y.values

print("=" * 50)
print("Generating Out-Of-Fold Predictions...")
print("=" * 50)
oof_xgb, oof_lgbm, oof_mlp = get_oof_predictions(X_full_scaled, Y_arr)

Generating Out-Of-Fold Predictions...

--- Fold 1/5 ---
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step
  Avg Ensemble AUC (fold): 0.8422

--- Fold 2/5 ---
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step
  Avg Ensemble AUC (fold): 0.8652

--- Fold 3/5 ---


1/5 ━━━━━━━━━━━━━━━━━━━━ 1s 260ms/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 71ms/step
  Avg Ensemble AUC (fold): 0.8533

--- Fold 4/5 ---
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step
  Avg Ensemble AUC (fold): 0.8228

--- Fold 5/5 ---
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step
  Avg Ensemble AUC (fold): 0.7658

✅ OOF Generation Complete
   Mean Fold AUC: 0.8299 ± 0.0349


In [9]:
# Stack OOF predictions as new features
meta_X_train = np.column_stack([oof_xgb, oof_lgbm, oof_mlp])
meta_y_train = Y_arr

print("Meta-learner input shape:", meta_X_train.shape)

# Logistic Regression as meta-learner (interpretable + avoids overfitting)
meta_learner = LogisticRegression(random_state=42, max_iter=1000)
meta_learner.fit(meta_X_train, meta_y_train)

print("\nMeta-learner weights (XGB | LGBM | MLP):")
print(np.round(meta_learner.coef_[0], 4))

Meta-learner input shape: (768, 3)

Meta-learner weights (XGB | LGBM | MLP):
[0.4941 0.7242 3.4829]


In [10]:
print("\nTraining final base models on full SMOTE train data...")

# XGBoost
final_xgb = build_xgb()
final_xgb.fit(X_train_sm, y_train_sm, verbose=False)
test_xgb = final_xgb.predict_proba(X_test_scaled)[:, 1]

# LightGBM
final_lgbm = build_lgbm()
final_lgbm.fit(X_train_sm, y_train_sm)
test_lgbm = final_lgbm.predict_proba(X_test_scaled)[:, 1]

# MLP
final_mlp = build_mlp(X_train_sm.shape[1])
early_stop = EarlyStopping(monitor='val_loss', patience=15,
                            restore_best_weights=True)
final_mlp.fit(X_train_sm, y_train_sm,
              epochs=150, batch_size=32,
              validation_split=0.1,
              callbacks=[early_stop], verbose=0)
test_mlp = final_mlp.predict(X_test_scaled).flatten()

print("✅ All base models trained")


Training final base models on full SMOTE train data...
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step
✅ All base models trained


In [11]:
# ── Method A: Stacking (Meta-Learner) ────────────────
meta_X_test       = np.column_stack([test_xgb, test_lgbm, test_mlp])
y_prob_stack      = meta_learner.predict_proba(meta_X_test)[:, 1]
y_pred_stack      = (y_prob_stack >= 0.5).astype(int)

# ── Method B: Weighted Average ────────────────────────
# Weights based on individual CV AUC performance
w_xgb, w_lgbm, w_mlp = 0.35, 0.35, 0.30
y_prob_weighted   = (w_xgb  * test_xgb +
                     w_lgbm * test_lgbm +
                     w_mlp  * test_mlp)
y_pred_weighted   = (y_prob_weighted >= 0.5).astype(int)

# ── Method C: Simple Average ──────────────────────────
y_prob_avg        = (test_xgb + test_lgbm + test_mlp) / 3
y_pred_avg        = (y_prob_avg >= 0.5).astype(int)

# ── Print All Results ─────────────────────────────────
print("\n" + "=" * 55)
print("         HYBRID ENSEMBLE — FINAL RESULTS")
print("=" * 55)

for name, y_pred, y_prob in [
    ("XGBoost (base)",         (test_xgb  >= 0.5).astype(int), test_xgb),
    ("LightGBM (base)",        (test_lgbm >= 0.5).astype(int), test_lgbm),
    ("MLP (base)",             (test_mlp  >= 0.5).astype(int), test_mlp),
    ("Simple Average",          y_pred_avg,      y_prob_avg),
    ("Weighted Average",        y_pred_weighted, y_prob_weighted),
    ("Stacking (Meta-Learner)", y_pred_stack,    y_prob_stack),
]:
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    f1  = f1_score(y_test, y_pred)
    print(f"{name:<28} | Acc: {acc:.4f} | AUC: {auc:.4f} | F1: {f1:.4f}")


         HYBRID ENSEMBLE — FINAL RESULTS
XGBoost (base)               | Acc: 0.7338 | AUC: 0.8161 | F1: 0.6306
LightGBM (base)              | Acc: 0.7662 | AUC: 0.8213 | F1: 0.6786
MLP (base)                   | Acc: 0.7403 | AUC: 0.7880 | F1: 0.6491
Simple Average               | Acc: 0.7532 | AUC: 0.8157 | F1: 0.6724
Weighted Average             | Acc: 0.7468 | AUC: 0.8167 | F1: 0.6609
Stacking (Meta-Learner)      | Acc: 0.7662 | AUC: 0.7972 | F1: 0.6604
